# AMM Simulator — Linear-Impact AMM

This notebook runs the Linear-Impact AMM simulation:

1. Imports & JAX verification
2. Single episode rollout
3. Batch rollout
4. Compare two pools with different spreads

## 1. Imports

In [7]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import importlib
import amm_sim.scoring
import amm_sim.spec
import amm_sim.types
import amm_sim.engine
import amm_sim.env
import amm_sim.amms.linear

importlib.reload(amm_sim.scoring)
importlib.reload(amm_sim.spec)
importlib.reload(amm_sim.types)
importlib.reload(amm_sim.amms.linear)
importlib.reload(amm_sim.engine)
importlib.reload(amm_sim.env)

from amm_sim.types import SimParams
from amm_sim.env import make_env
from amm_sim.amms.linear import (
    LINEAR_AMM, LinearParams, LinearState,
    linear_arb_solver, linear_edge,
    linear_route_two_pools,
    verify_jax_compatibility
)

print("JAX version :", jax.__version__)
print("JAX devices :", jax.devices())

JAX version : 0.9.2
JAX devices : [CpuDevice(id=0)]


## 2. JAX Verification

In [8]:
verify_jax_compatibility()

AssertionError: arb edge (Z+<0) should be negative

## 3. Simulation Parameters

In [ ]:
# Two identical Linear AMMs
# lam_pp / lam_mm: book depth (larger = less price impact)
# init_z_plus / init_z_minus: initial ask/bid spread
params = LinearParams(
    lam_pp=0.01,
    lam_mm=0.01,
    lam_pm=0.005,
    lam_mp=0.005,
    init_z_plus=0.3,
    init_z_minus=0.3,
)

sim_params = SimParams(
    sigma=0.001,
    num_steps=1_000,
    max_orders=16,
    lam=0.8,
    mu=1.0,
    sigma_ln=0.5,
    phi=0.0,
)

env = make_env(
    amm_specs   = [LINEAR_AMM, LINEAR_AMM],
    amm_params  = [params, params],
    arb_solvers = [linear_arb_solver, linear_arb_solver],
    route_fn    = linear_route_two_pools,
    edge_fns    = [linear_edge, linear_edge],
)

print("Environment ready.")

## 4. Single Episode Rollout

In [ ]:
key = jax.random.PRNGKey(0)
final_state, traj = env.rollout(key, sim_params)

arb_edge    = np.array(traj.arb_edge)
retail_edge = np.array(traj.retail_edge)
total_edge  = np.array(traj.total_edge)
fair_price  = np.array(traj.fair_price)

print(f"Episode summary:")
print(f"  Total edge  : {total_edge.sum():.4f}")
print(f"  Arb edge    : {arb_edge.sum():.4f}  (should be < 0)")
print(f"  Retail edge : {retail_edge.sum():.4f}  (should be > 0)")
print(f"  Arb always <= 0: {bool((arb_edge <= 1e-6).all())}")

# Final pool state
pool0 = final_state.amm_states[0]
print(f"\nFinal pool 0 state:")
print(f"  Z+  = {float(pool0.z_plus):.4f}  (ask spread)")
print(f"  Z-  = {float(pool0.z_minus):.4f}  (bid spread)")
print(f"  X   = {float(pool0.x):.4f}  (inventory)")
print(f"  Y   = {float(pool0.y):.4f}  (cash)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Single Episode — Two Identical Linear AMMs", fontsize=13)

steps = np.arange(len(fair_price))

axes[0, 0].plot(steps, fair_price, color="steelblue", lw=1)
axes[0, 0].set_title("Fair Price (GBM)")
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Price")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].bar(steps, retail_edge, color="seagreen", alpha=0.6, label="Retail", width=1)
axes[0, 1].bar(steps, arb_edge,    color="tomato",   alpha=0.6, label="Arb",    width=1)
axes[0, 1].set_title("Per-Step Edge")
axes[0, 1].set_xlabel("Step")
axes[0, 1].set_ylabel("Edge")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(steps, np.cumsum(total_edge),  color="navy",     lw=1.5, label="Total")
axes[1, 0].plot(steps, np.cumsum(arb_edge),    color="tomato",   lw=1,   label="Arb",    linestyle="--")
axes[1, 0].plot(steps, np.cumsum(retail_edge), color="seagreen", lw=1,   label="Retail", linestyle="--")
axes[1, 0].axhline(0, color="black", lw=0.8, linestyle=":")
axes[1, 0].set_title("Cumulative Edge")
axes[1, 0].set_xlabel("Step")
axes[1, 0].set_ylabel("Cumulative Edge")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(total_edge, bins=40, color="slategray", alpha=0.8, edgecolor="white")
axes[1, 1].axvline(0, color="black", lw=1, linestyle=":")
axes[1, 1].set_title("Per-Step Total Edge Distribution")
axes[1, 1].set_xlabel("Edge")
axes[1, 1].set_ylabel("Count")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Batch Rollout (200 episodes)

In [ ]:
import time

N_EPISODES = 200
keys = jax.random.split(jax.random.PRNGKey(42), N_EPISODES)

# Warm up JIT
_ = env.batch_rollout(keys[:4], sim_params)
jax.block_until_ready(_)

t0 = time.time()
final_states, trajs = env.batch_rollout(keys, sim_params)
jax.block_until_ready((final_states, trajs))
t1 = time.time()

total_steps = N_EPISODES * sim_params.num_steps
print(f"{N_EPISODES} episodes × {sim_params.num_steps} steps = {total_steps:,} total steps")
print(f"Wall time : {t1-t0:.3f}s  ({total_steps/(t1-t0):,.0f} steps/sec)")

batch_arb    = np.array(trajs.arb_edge)
batch_retail = np.array(trajs.retail_edge)
batch_total  = np.array(trajs.total_edge)

ep_arb    = batch_arb.sum(axis=1)
ep_retail = batch_retail.sum(axis=1)
ep_total  = batch_total.sum(axis=1)

print()
print(f"Arb edge always <= 0: {bool((batch_arb <= 1e-6).all())}")
print()
print(f"{'Metric':<25} {'Mean':>10}  {'Std':>10}  {'Min':>10}  {'Max':>10}")
print("-" * 68)
for name, arr in [("Total edge / episode", ep_total),
                  ("Arb edge / episode",   ep_arb),
                  ("Retail edge / episode",ep_retail)]:
    print(f"{name:<25} {arr.mean():>10.4f}  {arr.std():>10.4f}  "
          f"{arr.min():>10.4f}  {arr.max():>10.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Batch Rollout — {N_EPISODES} Episodes × {sim_params.num_steps} Steps", fontsize=13)

axes[0].hist(ep_total, bins=30, color="navy", alpha=0.8, edgecolor="white")
axes[0].axvline(ep_total.mean(), color="red", lw=1.5, linestyle="--",
                label=f"Mean={ep_total.mean():.3f}")
axes[0].set_title("Total Edge per Episode")
axes[0].set_xlabel("Total Edge")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(ep_arb, ep_retail, alpha=0.5, color="steelblue", s=20)
axes[1].set_title("Arb Edge vs Retail Edge")
axes[1].set_xlabel("Arb Edge (negative)")
axes[1].set_ylabel("Retail Edge (positive)")
axes[1].grid(True, alpha=0.3)

mean_cum = batch_total.mean(axis=0).cumsum()
axes[2].plot(mean_cum, color="navy", lw=1.5, label="Mean cumulative edge")
axes[2].axhline(0, color="black", lw=0.8, linestyle=":")
axes[2].set_title("Mean Cumulative Edge")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Cumulative Edge")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Compare Two Pools with Different Spreads

Pool 0 (agent): tighter spread Z+ = 0.2  
Pool 1 (normalizer): wider spread Z+ = 0.3

In [ ]:
params_tight = LinearParams(
    lam_pp=0.01, lam_mm=0.01,
    lam_pm=0.005, lam_mp=0.005,
    init_z_plus=0.2,    # tighter ask spread
    init_z_minus=0.2,
)
params_wide = LinearParams(
    lam_pp=0.01, lam_mm=0.01,
    lam_pm=0.005, lam_mp=0.005,
    init_z_plus=0.3,    # wider ask spread (normalizer)
    init_z_minus=0.3,
)

env2 = make_env(
    amm_specs   = [LINEAR_AMM, LINEAR_AMM],
    amm_params  = [params_tight, params_wide],
    arb_solvers = [linear_arb_solver, linear_arb_solver],
    route_fn    = linear_route_two_pools,
    edge_fns    = [linear_edge, linear_edge],
)

keys2 = jax.random.split(jax.random.PRNGKey(7), 200)
final2, trajs2 = env2.batch_rollout(keys2, sim_params)
jax.block_until_ready((final2, trajs2))

ep_total2 = np.array(trajs2.total_edge).sum(axis=1)
ep_arb2   = np.array(trajs2.arb_edge).sum(axis=1)
ep_ret2   = np.array(trajs2.retail_edge).sum(axis=1)

print("Pool 0 (tight spread Z+=0.2) vs Pool 1 (wide spread Z+=0.3)")
print(f"  Mean total edge : {ep_total2.mean():.4f}")
print(f"  Mean arb edge   : {ep_arb2.mean():.4f}")
print(f"  Mean retail edge: {ep_ret2.mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(ep_total,  bins=30, alpha=0.6, color="steelblue",
        label=f"Z+=0.3 baseline  mean={ep_total.mean():.2f}", edgecolor="white")
ax.hist(ep_total2, bins=30, alpha=0.6, color="seagreen",
        label=f"Z+=0.2 agent     mean={ep_total2.mean():.2f}", edgecolor="white")
ax.axvline(ep_total.mean(),  color="steelblue", lw=2, linestyle="--")
ax.axvline(ep_total2.mean(), color="seagreen",  lw=2, linestyle="--")
ax.set_title("Total Edge per Episode: Tight vs Wide Spread")
ax.set_xlabel("Total Edge")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()